In [ ]:
from google.colab import drive
import os
import h5py
import pandas as pd

drive.mount('/content/drive')

# Replication - Original Dataset

## Import Libraries

In [ ]:
# !pip install pycox torchtuples h5py

In [ ]:
# !pip install lifelines

In [ ]:
# !pip install --upgrade scikit-survival

In [ ]:
import h5py
import torch
import numpy as np
import pandas as pd
import os
import torch.nn as nn
from pycox.models import CoxPH
import torchtuples as tt
from pycox.evaluation import EvalSurv
from lifelines import CoxPHFitter
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Load Original Data

In [ ]:
np.random.seed(48764388)
_ = torch.manual_seed(48764388)

In [ ]:
# =====================================================================
# Load the author's .h5 data file from Google Drive
# - metabric_IHC4_clinical_train_test.h5
# =====================================================================

file_path = '/content/drive/MyDrive/metabric_IHC4_clinical_train_test.h5'

# Check if the file exists
if os.path.exists(file_path):
    with h5py.File(file_path, 'r') as f:
        # Extract Train / Test datasets
        x_train = f['train']['x'][:].astype('float32')
        e_train = f['train']['e'][:].astype('int32')
        t_train = f['train']['t'][:].astype('float32')

        x_test = f['test']['x'][:].astype('float32')
        e_test = f['test']['e'][:].astype('int32')
        t_test = f['test']['t'][:].astype('float32')

    print("✅ .h5 data file successfully loaded!")
    print(f"- Number of train patients: {x_train.shape[0]} / Number of input features: {x_train.shape[1]}")
    print(f"- Number of test patients: {x_test.shape[0]}")

    # Feature names matching Table 1 and dataset description
    column_names = [
        'MKI67',
        'EGFR',
        'PGR',
        'ERBB2',
        'Hormone',
        'Radio',
        'Chemo',
        'ER-pos',
        'Age'
    ]

    # Create a pandas DataFrame for train data inspection
    df_train = pd.DataFrame(x_train, columns=column_names)
    df_train['Event (e)'] = e_train
    df_train['Time (t)'] = t_train

    # Inspect the top 5 patient records
    print("\n📊 Top 5 samples of Train data:")
    display(df_train.head())
else:
    print(f"⚠️ File '{file_path}' not found. Please verify the file path.")

In [ ]:
# Format target variables as (duration, event) tuples
y_train = (t_train, e_train)
y_test = (t_test, e_test)

# Split validation set (20% of the training data)
val_size = int(len(x_train) * 0.2)
x_val = x_train[:val_size]
y_val = (t_train[:val_size], e_train[:val_size])

x_train_sub = x_train[val_size:]
y_train_sub = (t_train[val_size:], e_train[val_size:])

## Model Development (DeepSurv)

In [ ]:
# =====================================================================
# Build DeepSurv Neural Network Architecture (Tuned)
# =====================================================================
in_features = x_train.shape[1]   # Number of input features (9)
num_nodes = [32, 32]             # Hidden layers
out_features = 1                 # Risk score output
dropout = 0.1

# Apply SELU activation function to enhance neural network performance
net = tt.practical.MLPVanilla(
    in_features,
    num_nodes,
    out_features,
    batch_norm=True,
    dropout=dropout,
    activation=nn.SELU # Apply SELU activation function
)

# Adjust learning rate (0.01 -> 0.001) for finer optimization
optimizer = tt.optim.Adam(lr=0.001, weight_decay=1e-4)
model = CoxPH(net, optimizer)

print("✅ Model initialized with tuned hyperparameters!")

In [ ]:
# =====================================================================
# DeepSurv Model Training
# =====================================================================
batch_size = 256
epochs = 100
callbacks = [tt.callbacks.EarlyStopping(patience=10)]

print("\n Starting DeepSurv model training...")
log = model.fit(
    x_train_sub,
    y_train_sub,
    batch_size=batch_size,
    epochs=epochs,
    callbacks=callbacks,
    val_data=(x_val, y_val),
    verbose=True
)

In [ ]:
# =====================================================================
# [Key Result] Evaluate and Display Test C-index (Concordance Index)
# =====================================================================
# Predict survival probabilities/hazards on the test dataset
_ = model.compute_baseline_hazards()
surv = model.predict_surv_df(x_test)

# Calculate C-index using the survival evaluation class (EvalSurv)
ev = EvalSurv(surv, t_test, e_test, censor_surv='km')
c_index = ev.concordance_td()

print("\n" + "="*50)
print(f"[Replication Result] METABRIC Test C-index: {c_index:.4f}")
print("="*50)

## Model Development (CPH)

In [ ]:
# Convert to pandas DataFrames (Train & Test)
df_train = pd.DataFrame(x_train)
df_train['t'] = t_train
df_train['e'] = e_train

df_test = pd.DataFrame(x_test)
df_test['t'] = t_test
df_test['e'] = e_test

# Fit and evaluate the Cox Proportional Hazards (CPH) model
cph = CoxPHFitter()
cph.fit(df_train, duration_col='t', event_col='e')

# Measure the Test C-index
cph_c_index = cph.score(df_test, scoring_method="concordance_index")
print(f"CPH Test C-index: {cph_c_index:.4f}")

## Model Development (RSF)

In [ ]:
y_train_struct = np.array(
    list(zip(e_train.astype(bool), t_train)),
    dtype=[('Status', '?'), ('Survival_in_days', '<f8')]
)

y_test_struct = np.array(
    list(zip(e_test.astype(bool), t_test)),
    dtype=[('Status', '?'), ('Survival_in_days', '<f8')]
)

print("Structured array conversion for RSF completed!")

# -------------------------------------------------------------------------
# Initialize and Train Random Survival Forest (RSF) Model
# -------------------------------------------------------------------------
rsf = RandomSurvivalForest(
    n_estimators=100,        # Number of trees (default: 100)
    min_samples_split=10,    # Minimum samples required to split a node
    min_samples_leaf=15,     # Minimum samples required at a leaf node
    max_features="sqrt",     # Number of features to consider when looking for the best split
    n_jobs=-1,               # Parallel processing (speed optimization)
    random_state=42
)

rsf.fit(x_train, y_train_struct)
print("RSF model training completed")

# -------------------------------------------------------------------------
# Model Evaluation (Compute Test C-index only)
# -------------------------------------------------------------------------
c_index_test = rsf.score(x_test, y_test_struct)

print("\n" + "="*40)
print(f"RSF Performance:")
print(f"Test C-index : {c_index_test:.4f}")
print("="*40)

# Research Question 1: Does DeepSurv perform well on a domain-informed dataset?

# Test Models - Cirrhosis Dataset

## Load Cirrhosis Data

In [ ]:
# =====================================================================
# Load Cirrhosis Clinical Dataset
# =====================================================================
import os
import pandas as pd

# Define file path (adjust the path if loaded from Google Drive)
file_path = '/content/drive/MyDrive/cirrhosis.csv'

# Check if the file exists and load into DataFrame
if os.path.exists(file_path):
    df_ch = pd.read_csv(file_path)
else:
    # Fallback to local root path
    df_ch = pd.read_csv('cirrhosis.csv')

# Inspect the first 3 patient records
print("📊 First 5 samples of Cirrhosis dataset:")
df_ch.head(5)

In [ ]:
# One-Hot-Encoding
binary_cols = ["Ascites", "Hepatomegaly", "Spiders"]

# 'Y' -> 1, 'N' -> 0
df_ch_dum = df_ch.copy()

df_ch_dum = pd.get_dummies(
    df_ch_dum, columns=binary_cols, drop_first=True, dtype=float
)

df_ch_dum.head(5)

In [ ]:
# =====================================================================
# 1. Feature Engineering & Align to Table 4 Single Unified View
# =====================================================================
import numpy as np
import pandas as pd
from IPython.display import display

# 1-1. Binary Encoding (Y -> 1.0, N -> 0.0)
df_ch_prep = df_ch.copy()
for col in ["Ascites", "Hepatomegaly", "Spiders"]:
    if col in df_ch_prep.columns:
        df_ch_prep[f"{col}_Y"] = df_ch_prep[col].map({"Y": 1.0, "N": 0.0})

# 1-2. Generate Nonlinear Stage-Weighted Risk (X1)
alpha = np.log(2)
stage_weight = np.exp(alpha * (df_ch_prep["Stage"].astype(float) - 1.0))
biomarker_risk = (
    np.log(df_ch_prep["Cholesterol"].astype(float) + 1.0)
    + np.log(df_ch_prep["Tryglicerides"].astype(float) + 1.0)
    - np.log(df_ch_prep["Albumin"].astype(float) + 1e-5)
)
df_ch_prep["Stage_Weighted_Risk"] = stage_weight * biomarker_risk

# 1-3. Target Processing (Status binarization & missing value dropping)
df_ch_prep["Status"] = df_ch_prep["Status"].map({"D": 1.0, "C": 0.0, "CL": 0.0})

ordered_cols = [
    "Ascites_Y",
    "Hepatomegaly_Y",
    "Spiders_Y",
    "Bilirubin",
    "Copper",
    "Prothrombin",
    "Alk_Phos",
    "SGOT",
    "Platelets",
    "Stage_Weighted_Risk",
    "N_Days",
    "Status"
]

# Drop rows with missing values in the required columns
df_ch_clean = df_ch_prep[ordered_cols].dropna().reset_index(drop=True)

# 1-4. Rename to Table 4 exact headers
rename_headers = {
    "Ascites_Y": "Ascites (Y)",
    "Hepatomegaly_Y": "Hepato (Y)",
    "Spiders_Y": "Spiders (Y)",
    "Alk_Phos": "Alk Phos",
    "Stage_Weighted_Risk": "Stage-Weighted Risk (X1)",
    "N_Days": "Duration (T)",
    "Status": "Event (δ)"
}

df_ch_table4_single = df_ch_clean.rename(columns=rename_headers).copy()
df_ch_table4_single["Event (δ)"] = df_ch_table4_single["Event (δ)"].astype(int)

df_ch_table4_single["Stage-Weighted Risk (X1)"] = df_ch_table4_single["Stage-Weighted Risk (X1)"].round(2)
df_ch_table4_single["Duration (T)"] = df_ch_table4_single["Duration (T)"].astype(int)

print("=" * 80)
print("Sample Instances of the Domain-Engineered Cirrhosis Patient Survival Dataset")
print("=" * 80)
display(df_ch_table4_single.head(5))

## Test Models on the domain-guided cirrhosis survival dataset

In [ ]:
# 0. Set random seed for reproducibility
SEED = 48764388
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =====================================================================
# 1. Feature Engineering and Multivariate Preprocessing
# =====================================================================
df_ch_wt = df_ch_dum.copy()

# 1-1. Generate nonlinear Stage-Weighted Risk feature
alpha = np.log(2)
stage_weight = np.exp(alpha * (df_ch_wt["Stage"] - 1.0))
biomarker_risk = (
    np.log(df_ch_wt["Cholesterol"] + 1.0)
    + np.log(df_ch_wt["Tryglicerides"] + 1.0)
    - np.log(df_ch_wt["Albumin"] + 1e-5)
)
df_ch_wt["Stage_Weighted_Risk"] = stage_weight * biomarker_risk

# 1-2. Define final 10-feature set
columns_to_keep = [
    "N_Days",
    "Status",
    "Ascites_Y",
    "Hepatomegaly_Y",
    "Spiders_Y",
    "Stage_Weighted_Risk",
    "Bilirubin",
    "Copper",
    "Prothrombin",
    "Alk_Phos",
    "SGOT",
    "Platelets"
]

df_ch_final = df_ch_wt[[c for c in columns_to_keep if c in df_ch_wt.columns]].copy()

# 1-3. Binarize Status (D: 1.0, C/CL: 0.0) and handle missing values
df_ch_final["Status"] = df_ch_final["Status"].map({"D": 1.0, "C": 0.0, "CL": 0.0})
df_ch_final = df_ch_final.dropna().reset_index(drop=True)

df_ch_final["Status"] = df_ch_final["Status"].astype("float32")
df_ch_final["N_Days"] = df_ch_final["N_Days"].astype("float32")

# =====================================================================
# 2. Stratified Data Splitting and Feature Scaling
# =====================================================================
feature_cols = [col for col in df_ch_final.columns if col not in ["N_Days", "Status"]]
X = df_ch_final[feature_cols].values.astype("float32")
durations = df_ch_final["N_Days"].values.astype("float32")
events = df_ch_final["Status"].values.astype("float32")

# Step 1: Split into Train+Val (85%) vs Test (15%)
X_train_full, X_test, d_train_full, d_test, e_train_full, e_test = train_test_split(
    X, durations, events, test_size=0.15, random_state=SEED, stratify=events
)

# Step 2: Split Train+Val into Train (70%) vs Val (15%) - For DeepSurv Early Stopping
X_train, X_val, d_train, d_val, e_train, e_val = train_test_split(
    X_train_full, d_train_full, e_train_full, test_size=0.17647, random_state=SEED, stratify=e_train_full
)

# Standardize features (Fit on Train only to prevent data leakage)
scaler = StandardScaler()
X_train_full_scaled = scaler.fit_transform(X_train_full)
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# =====================================================================
# 3. [Model 1] Cox Proportional Hazards (CPH)
# =====================================================================
df_train_cph = pd.DataFrame(X_train_full_scaled, columns=feature_cols)
df_train_cph['t'] = d_train_full
df_train_cph['e'] = e_train_full

df_test_cph = pd.DataFrame(X_test_scaled, columns=feature_cols)
df_test_cph['t'] = d_test
df_test_cph['e'] = e_test

cph = CoxPHFitter(penalizer=0.05)
cph.fit(df_train_cph, duration_col='t', event_col='e')

# Predict partial hazard log-ratio and compute Concordance Index
cph_train_risk = cph.predict_log_partial_hazard(df_train_cph).values.flatten()
cph_test_risk = cph.predict_log_partial_hazard(df_test_cph).values.flatten()

cph_train_c = concordance_index_censored(e_train_full.astype(bool), d_train_full, cph_train_risk)[0]
cph_test_c = concordance_index_censored(e_test.astype(bool), d_test, cph_test_risk)[0]

# =====================================================================
# 4. [Model 2] Random Survival Forest (RSF)
# =====================================================================
# Construct structured survival target array for scikit-survival
y_train_struct = np.array(
    list(zip(e_train_full.astype(bool), d_train_full)),
    dtype=[('Status', '?'), ('Survival_in_days', '<f8')]
)
y_test_struct = np.array(
    list(zip(e_test.astype(bool), d_test)),
    dtype=[('Status', '?'), ('Survival_in_days', '<f8')]
)

rsf = RandomSurvivalForest(
    n_estimators=100,
    min_samples_split=10,
    min_samples_leaf=15,
    max_features="sqrt",
    n_jobs=-1,
    random_state=SEED
)
rsf.fit(X_train_full_scaled, y_train_struct)

rsf_train_c = rsf.score(X_train_full_scaled, y_train_struct)
rsf_test_c = rsf.score(X_test_scaled, y_test_struct)

# =====================================================================
# 5. [Model 3] DeepSurv (Deep Neural Network)
# =====================================================================
y_train_deep = (d_train, e_train)
y_val_deep = (d_val, e_val)
val_data = (X_val_scaled, y_val_deep)

in_features = X_train_scaled.shape[1]
net = tt.practical.MLPVanilla(
    in_features,
    [32, 16],
    out_features=1,
    batch_norm=True,
    dropout=0.2,
    activation=nn.SELU
)

model_deep = CoxPH(net, tt.optim.Adam(lr=0.002, weight_decay=1e-3))
callbacks = [tt.callbacks.EarlyStopping(patience=20)]

log = model_deep.fit(
    X_train_scaled, y_train_deep,
    batch_size=64,
    epochs=200,
    callbacks=callbacks,
    val_data=val_data,
    verbose=False
)

# Predict risk scores and calibrate risk score direction
risk_train_deep = model_deep.predict(X_train_scaled).flatten()
risk_test_deep = model_deep.predict(X_test_scaled).flatten()

if risk_test_deep[e_test == 1].mean() < risk_test_deep[e_test == 0].mean():
    risk_train_deep = -risk_train_deep
    risk_test_deep = -risk_test_deep

deepsurv_train_c = concordance_index_censored(e_train.astype(bool), d_train, risk_train_deep)[0]
deepsurv_test_c = concordance_index_censored(e_test.astype(bool), d_test, risk_test_deep)[0]

# =====================================================================
# 6. Final Performance Benchmark Table
# =====================================================================
summary_data = {
    "Model": ["Cox Proportional Hazards (CPH)", "Random Survival Forest (RSF)", "DeepSurv (Neural Network)"],
    "Model Family": ["Semiparametric / Linear", "Nonlinear Tree Ensemble", "Deep Neural Network"],
    "Train C-Index": [f"{cph_train_c:.4f}", f"{rsf_train_c:.4f}", f"{deepsurv_train_c:.4f}"],
    "Test C-Index": [f"{cph_test_c:.4f}", f"{rsf_test_c:.4f}", f"{deepsurv_test_c:.4f}"]
}

df_results = pd.DataFrame(summary_data)

print("\n" + "="*70)
print(f"📊 [Cirrhosis Dataset] Survival Benchmark Comparison (Features: {in_features})")
print("="*70)
print(df_results.to_string(index=False))
print("="*70)

# Test Models - German Credit Dataset

## Load German Bank Data

In [ ]:
# =====================================================================
# Load German Credit Dataset (german.data)
# =====================================================================
import os
import pandas as pd

# Set Pandas options to display all columns without truncation
pd.set_option('display.max_columns', None)       # Display all columns
pd.set_option('display.width', 1000)             # Increase display width to prevent line wrapping
pd.set_option('display.max_colwidth', None)      # Do not truncate content inside cells

# Define path (adjust if stored inside Google Drive)
file_path_german = '/content/drive/MyDrive/german.data'

if not os.path.exists(file_path_german):
    file_path_german = 'german.data'

# Standard semantic column names according to UCI attribute specification
german_cols = [
    "Status_Checking_Acc",
    "Duration_in_month",
    "Credit_History",
    "Purpose",
    "Credit_Amount",
    "Savings_Account",
    "Present_Employment_Since",
    "Installment_Rate",
    "Personal_Status_Sex",
    "Other_Debtors",
    "Present_Residence_Since",
    "Property",
    "Age_in_years",
    "Other_Installment_Plans",
    "Housing",
    "Existing_Credits",
    "Job",
    "Num_People_Liable",
    "Telephone",
    "Foreign_Worker",
    "Credit_Risk"  # Target (1 = Good, 2 = Bad)
]

# Load using whitespace delimiter
df_german = pd.read_csv(file_path_german, sep=r'\s+', header=None, names=german_cols)

print(f"📊 Successfully loaded German Credit Data (Rows: {df_german.shape[0]}, Columns: {df_german.shape[1]}):")
display(df_german.head(5))

In [ ]:
# =====================================================================
# Load and Encode German Credit Dataset (german.data)
# =====================================================================
import os
import pandas as pd

# Set Pandas options to display all columns and prevent truncation
pd.set_option('display.max_columns', None)       # Display all columns
pd.set_option('display.width', 1000)             # Increase display width to prevent line wrapping
pd.set_option('display.max_colwidth', None)      # Do not truncate content inside cells

# Define path (adjust if stored inside Google Drive)
file_path_german = '/content/drive/MyDrive/german.data'

if not os.path.exists(file_path_german):
    file_path_german = 'german.data'

# Standard semantic column names according to UCI attribute specification
german_cols = [
    "Status_Checking_Acc",
    "Duration_in_month",
    "Credit_History",
    "Purpose",
    "Credit_Amount",
    "Savings_Account",
    "Present_Employment_Since",
    "Installment_Rate",
    "Personal_Status_Sex",
    "Other_Debtors",
    "Present_Residence_Since",
    "Property",
    "Age_in_years",
    "Other_Installment_Plans",
    "Housing",
    "Existing_Credits",
    "Job",
    "Num_People_Liable",
    "Telephone",
    "Foreign_Worker",
    "Credit_Risk"  # Target (1 = Good, 2 = Bad)
]

# Load using whitespace delimiter
df_german = pd.read_csv(file_path_german, sep=r'\s+', header=None, names=german_cols)

# Copy dataframe for encoding
df_encoded = df_german.copy()

# Define mapping dictionaries for categorical features
cs_map = {"A11": 1.0, "A12": 2.0, "A13": 3.0, "A14": 4.0}
ss_map = {"A61": 1.0, "A62": 2.0, "A63": 3.0, "A64": 4.0, "A65": 1.0}
es_map = {"A71": 0.0, "A72": 1.0, "A73": 2.0, "A74": 3.0, "A75": 4.0}
ps_map = {"A121": 4.0, "A122": 3.0, "A123": 2.0, "A124": 1.0}
gs_map = {"A101": 1.0, "A102": 2.0, "A103": 3.0}

# Apply mappings to encoded dataframe
df_encoded["Status_Checking_Acc"] = df_encoded["Status_Checking_Acc"].map(cs_map).fillna(1.0)
df_encoded["Savings_Account"] = df_encoded["Savings_Account"].map(ss_map).fillna(1.0)
df_encoded["Present_Employment_Since"] = df_encoded["Present_Employment_Since"].map(es_map).fillna(0.0)
df_encoded["Property"] = df_encoded["Property"].map(ps_map).fillna(1.0)
df_encoded["Other_Debtors"] = df_encoded["Other_Debtors"].map(gs_map).fillna(1.0)

print(f"📊 Successfully loaded and mapped German Credit Data (Rows: {df_encoded.shape[0]}, Columns: {df_encoded.shape[1]}:")
display(df_encoded.head(5))

In [ ]:
# =====================================================================
# Domain Feature Engineering Pipeline (Final Unified Version)
# =====================================================================
import numpy as np
import pandas as pd
from IPython.display import display

df = df_german.copy()

# 1. Define Survival Analysis Target Variables (Time, Event)
# Event indicator: 1 = Good (0, Censored), 2 = Bad (1, Default Event)
df["Duration (Months) (T)"] = df["Duration_in_month"].astype(float)
df["Event (δ)"] = (df["Credit_Risk"] == 2).astype(int)

# 2. Domain Categorical Ordinal Score Mappings
cs_map = {"A11": 1.0, "A12": 2.0, "A13": 3.0, "A14": 4.0}
ss_map = {"A61": 1.0, "A62": 2.0, "A63": 3.0, "A64": 4.0, "A65": 1.0}
es_map = {"A71": 0.0, "A72": 1.0, "A73": 2.0, "A74": 3.0, "A75": 4.0}
ps_map = {"A121": 4.0, "A122": 3.0, "A123": 2.0, "A124": 1.0}
gs_map = {"A101": 1.0, "A102": 2.0, "A103": 3.0}

df["CS"] = df["Status_Checking_Acc"].map(cs_map).fillna(1.0)
df["SS"] = df["Savings_Account"].map(ss_map).fillna(1.0)
df["ES"] = df["Present_Employment_Since"].map(es_map).fillna(0.0)
df["PS"] = df["Property"].map(ps_map).fillna(1.0)
df["GS"] = df["Other_Debtors"].map(gs_map).fillna(1.0)

epsilon = 1e-5

# 3. Generate 4 Domain Features Based on Paper Formulations (Eq. 3 - Eq. 6)
# (1) Financial Burden Index (X1) - Eq. 3
df["Financial Burden Index (X1)"] = df["Installment_Rate"] * np.log(
    (df["Credit_Amount"] / (df["CS"] * df["SS"] + epsilon)) + 1.0
)

# (2) Asset and Employment Stability (X2) - Eq. 4
df["Asset and Employment Stability (X2)"] = df["PS"] * df["ES"]

# (3) Net Capacity per Dependent (X3) - Eq. 5
df["Net Capacity per Dependent (X3)"] = (
    df["Age_in_years"].astype(float) + df["Present_Residence_Since"].astype(float)
) / df["Num_People_Liable"].astype(float)

# (4) Guarantor-Adjusted Credit (X4) - Eq. 6
df["Guarantor-Adjusted Credit (X4)"] = (
    df["Credit_Amount"].astype(float) / (df["GS"] + epsilon)
)

# 4. Construct Final Feature Matrix for Survival Analysis
final_cols = [
    "Financial Burden Index (X1)",
    "Asset and Employment Stability (X2)",
    "Net Capacity per Dependent (X3)",
    "Guarantor-Adjusted Credit (X4)",
    "Duration (Months) (T)",
    "Event (δ)"
]

df_domain_final = df[final_cols].copy()

df_domain_final["Financial Burden Index (X1)"] = df_domain_final["Financial Burden Index (X1)"].round(2)
df_domain_final["Guarantor-Adjusted Credit (X4)"] = df_domain_final["Guarantor-Adjusted Credit (X4)"].round(2)
df_domain_final["Duration (Months) (T)"] = df_domain_final["Duration (Months) (T)"].astype(int)

display(df_domain_final.head(5))

## Test Models on the domain-guided German credit dataset

In [ ]:
# =====================================================================
# 0. Set Random Seed for Reproducibility
# =====================================================================
SEED = 48764388
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =====================================================================
# 1. Domain Feature Engineering Pipeline (Strict Eq. 3 ~ Eq. 6)
# =====================================================================
df = df_german.copy()

# 1-1. Define Target Variables (T: Duration in Months, Event: 1 for Bad, 0 for Good)
df["Duration (Months) (T)"] = df["Duration_in_month"].astype("float32")
df["Event (δ)"] = (df["Credit_Risk"] == 2).astype("float32")

# 1-2. Domain Categorical Ordinal Score Mappings (Paper Specification)
cs_map = {"A11": 1.0, "A12": 2.0, "A13": 3.0, "A14": 4.0}
ss_map = {"A61": 1.0, "A62": 2.0, "A63": 3.0, "A64": 4.0, "A65": 1.0}
es_map = {"A71": 0.0, "A72": 1.0, "A73": 2.0, "A74": 3.0, "A75": 4.0}
ps_map = {"A121": 4.0, "A122": 3.0, "A123": 2.0, "A124": 1.0}
gs_map = {"A101": 1.0, "A102": 2.0, "A103": 3.0}

df["CS"] = df["Status_Checking_Acc"].map(cs_map).fillna(1.0)
df["SS"] = df["Savings_Account"].map(ss_map).fillna(1.0)
df["ES"] = df["Present_Employment_Since"].map(es_map).fillna(0.0)
df["PS"] = df["Property"].map(ps_map).fillna(1.0)
df["GS"] = df["Other_Debtors"].map(gs_map).fillna(1.0)

epsilon = 1e-5

# 1-3. Generate 4 Domain Features
# Eq. 3: Financial Burden Index (X1)
df["Financial Burden Index (X1)"] = df["Installment_Rate"] * np.log(
    (df["Credit_Amount"] / (df["CS"] * df["SS"] + epsilon)) + 1.0
)

# Eq. 4: Asset and Employment Stability (X2)
df["Asset and Employment Stability (X2)"] = df["PS"] * df["ES"]

# Eq. 5: Net Capacity per Dependent (X3)
df["Net Capacity per Dependent (X3)"] = (
    df["Age_in_years"].astype(float) + df["Present_Residence_Since"].astype(float)
) / df["Num_People_Liable"].astype(float)

# Eq. 6: Guarantor-Adjusted Credit (X4)
df["Guarantor-Adjusted Credit (X4)"] = (
    df["Credit_Amount"].astype(float) / (df["GS"] + epsilon)
)

# 1-4. Construct Final Feature Matrix
feature_cols = [
    "Financial Burden Index (X1)",
    "Asset and Employment Stability (X2)",
    "Net Capacity per Dependent (X3)",
    "Guarantor-Adjusted Credit (X4)"
]
final_cols = feature_cols + ["Duration (Months) (T)", "Event (δ)"]

df_domain_final = df[final_cols].dropna().reset_index(drop=True)
display(df_domain_final.head(5))

# =====================================================================
# 2. Stratified Data Splitting & Feature Standardization
# =====================================================================
X = df_domain_final[feature_cols].values.astype("float32")
durations = df_domain_final["Duration (Months) (T)"].values.astype("float32")
events = df_domain_final["Event (δ)"].values.astype("float32")

# Step 1: Split into Train+Val (85%) vs Test (15%)
X_train_full, X_test, d_train_full, d_test, e_train_full, e_test = train_test_split(
    X, durations, events, test_size=0.15, random_state=SEED, stratify=events
)

# Step 2: Split Train+Val into Train (70%) vs Val (15%) for DeepSurv Early Stopping
X_train, X_val, d_train, d_val, e_train, e_val = train_test_split(
    X_train_full, d_train_full, e_train_full, test_size=0.17647, random_state=SEED, stratify=e_train_full
)

# Feature standardization (fitted strictly on training set to prevent leakage)
scaler = StandardScaler()
X_train_full_scaled = scaler.fit_transform(X_train_full)
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# =====================================================================
# 3. [Model 1] Cox Proportional Hazards (CPH)
# =====================================================================
df_train_cph = pd.DataFrame(X_train_full_scaled, columns=feature_cols)
df_train_cph['t'] = d_train_full
df_train_cph['e'] = e_train_full

df_test_cph = pd.DataFrame(X_test_scaled, columns=feature_cols)
df_test_cph['t'] = d_test
df_test_cph['e'] = e_test

cph = CoxPHFitter(penalizer=0.05)
cph.fit(df_train_cph, duration_col='t', event_col='e')

cph_train_risk = cph.predict_log_partial_hazard(df_train_cph).values.flatten()
cph_test_risk = cph.predict_log_partial_hazard(df_test_cph).values.flatten()

cph_train_c = concordance_index_censored(e_train_full.astype(bool), d_train_full, cph_train_risk)[0]
cph_test_c = concordance_index_censored(e_test.astype(bool), d_test, cph_test_risk)[0]

# =====================================================================
# 4. [Model 2] Random Survival Forest (RSF)
# =====================================================================
y_train_struct = np.array(
    list(zip(e_train_full.astype(bool), d_train_full)),
    dtype=[('Status', '?'), ('Survival_in_days', '<f8')]
)
y_test_struct = np.array(
    list(zip(e_test.astype(bool), d_test)),
    dtype=[('Status', '?'), ('Survival_in_days', '<f8')]
)

rsf = RandomSurvivalForest(
    n_estimators=100,
    min_samples_split=10,
    min_samples_leaf=15,
    max_features="sqrt",
    n_jobs=-1,
    random_state=SEED
)
rsf.fit(X_train_full_scaled, y_train_struct)

rsf_train_c = rsf.score(X_train_full_scaled, y_train_struct)
rsf_test_c = rsf.score(X_test_scaled, y_test_struct)

# =====================================================================
# 5. [Model 3] DeepSurv (Neural Network) - Calibrated Risk Score
# =====================================================================
y_train_deep = (d_train, e_train)
y_val_deep = (d_val, e_val)
val_data = (X_val_scaled, y_val_deep)

in_features = X_train_scaled.shape[1]
net = tt.practical.MLPVanilla(
    in_features,
    [32, 16],
    out_features=1,
    batch_norm=True,
    dropout=0.2,
    activation=nn.SELU
)

model_deep = CoxPH(net, tt.optim.Adam(lr=0.002, weight_decay=1e-3))
callbacks = [tt.callbacks.EarlyStopping(patience=20)]

log = model_deep.fit(
    X_train_scaled, y_train_deep,
    batch_size=64,
    epochs=200,
    callbacks=callbacks,
    val_data=val_data,
    verbose=False
)

# pycox outputs log-hazard (higher score = higher hazard)
# Passed directly to concordance_index_censored without erroneous sign flipping
risk_train_deep = model_deep.predict(X_train_scaled).flatten()
risk_test_deep = model_deep.predict(X_test_scaled).flatten()

deepsurv_train_c = concordance_index_censored(e_train.astype(bool), d_train, risk_train_deep)[0]
deepsurv_test_c = concordance_index_censored(e_test.astype(bool), d_test, risk_test_deep)[0]

# =====================================================================
# 6. Benchmark Performance Table
# =====================================================================
summary_data = {
    "Model": ["Cox Proportional Hazards (CPH)", "Random Survival Forest (RSF)", "DeepSurv (Neural Network)"],
    "Model Family": ["Semiparametric / Linear", "Nonlinear Tree Ensemble", "Deep Neural Network"],
    "Train C-Index": [f"{cph_train_c:.4f}", f"{rsf_train_c:.4f}", f"{deepsurv_train_c:.4f}"],
    "Test C-Index": [f"{cph_test_c:.4f}", f"{rsf_test_c:.4f}", f"{deepsurv_test_c:.4f}"]
}

df_results = pd.DataFrame(summary_data)

print("\n" + "="*70)
print(f"📊 [German Credit Dataset] Survival Benchmark Comparison (Features: {in_features})")
print("="*70)
print(df_results.to_string(index=False))
print("="*70)

# Research Question 2: Does DeepSurv perform better on a domain-informed dataset than on the raw dataset?

# Test Models - Cirrhosis Dataset

In [ ]:
# =====================================================================
# 0. Set random seed for reproducibility
# =====================================================================
SEED = 48764388
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =====================================================================
# 1. Strictly Selected Raw Features (Extracting 13 raw features used in engineering)
# =====================================================================
exact_raw_feature_cols = [
    "Ascites_Y",
    "Hepatomegaly_Y",
    "Spiders_Y",
    "Bilirubin",
    "Copper",
    "Prothrombin",
    "Alk_Phos",
    "SGOT",
    "Platelets",
    "Stage",
    "Cholesterol",
    "Tryglicerides",
    "Albumin"
]

target_cols = ["N_Days", "Status"]
columns_to_keep = target_cols + exact_raw_feature_cols

df_ch_raw_exact = df_ch_dum[[c for c in columns_to_keep if c in df_ch_dum.columns]].copy()

# Target binarization (D: 1.0, C/CL: 0.0)
df_ch_raw_exact["Status"] = df_ch_raw_exact["Status"].map({"D": 1.0, "C": 0.0, "CL": 0.0})

# Drop missing values and reset index to completely prevent NaN issues
df_ch_raw_exact = df_ch_raw_exact.dropna().reset_index(drop=True)

df_ch_raw_exact["Status"] = df_ch_raw_exact["Status"].astype("float32")
df_ch_raw_exact["N_Days"] = df_ch_raw_exact["N_Days"].astype("float32")

X = df_ch_raw_exact[exact_raw_feature_cols].values.astype("float32")
durations = df_ch_raw_exact["N_Days"].values.astype("float32")
events = df_ch_raw_exact["Status"].values.astype("float32")

# =====================================================================
# 2. Stratified Data Splitting & Feature Scaling (Identical splitting)
# =====================================================================
# Step 1: Split into Train+Val (85%) vs Test (15%)
X_train_full, X_test, d_train_full, d_test, e_train_full, e_test = train_test_split(
    X, durations, events, test_size=0.15, random_state=SEED, stratify=events
)

# Step 2: Split Train+Val into Train (70%) vs Val (15%)
X_train, X_val, d_train, d_val, e_train, e_val = train_test_split(
    X_train_full, d_train_full, e_train_full, test_size=0.17647, random_state=SEED, stratify=e_train_full
)

# Standardize features (fit based on training data)
scaler = StandardScaler()
X_train_full_scaled = scaler.fit_transform(X_train_full)
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# =====================================================================
# 3. [Model 1] Cox Proportional Hazards (CPH)
# =====================================================================
df_train_cph = pd.DataFrame(X_train_full_scaled, columns=exact_raw_feature_cols)
df_train_cph['t'] = d_train_full
df_train_cph['e'] = e_train_full

df_test_cph = pd.DataFrame(X_test_scaled, columns=exact_raw_feature_cols)
df_test_cph['t'] = d_test
df_test_cph['e'] = e_test

cph = CoxPHFitter(penalizer=0.05)
cph.fit(df_train_cph, duration_col='t', event_col='e')

cph_train_risk = cph.predict_log_partial_hazard(df_train_cph).values.flatten()
cph_test_risk = cph.predict_log_partial_hazard(df_test_cph).values.flatten()

cph_train_c = concordance_index_censored(e_train_full.astype(bool), d_train_full, cph_train_risk)[0]
cph_test_c = concordance_index_censored(e_test.astype(bool), d_test, cph_test_risk)[0]

# =====================================================================
# 4. [Model 2] Random Survival Forest (RSF)
# =====================================================================
y_train_struct = np.array(
    list(zip(e_train_full.astype(bool), d_train_full)),
    dtype=[('Status', '?'), ('Survival_in_days', '<f8')]
)
y_test_struct = np.array(
    list(zip(e_test.astype(bool), d_test)),
    dtype=[('Status', '?'), ('Survival_in_days', '<f8')]
)

rsf = RandomSurvivalForest(
    n_estimators=100,
    min_samples_split=10,
    min_samples_leaf=15,
    max_features="sqrt",
    n_jobs=-1,
    random_state=SEED
)
rsf.fit(X_train_full_scaled, y_train_struct)

rsf_train_c = rsf.score(X_train_full_scaled, y_train_struct)
rsf_test_c = rsf.score(X_test_scaled, y_test_struct)

# =====================================================================
# 5. [Model 3] DeepSurv (Deep Neural Network)
# =====================================================================
y_train_deep = (d_train, e_train)
y_val_deep = (d_val, e_val)
val_data = (X_val_scaled, y_val_deep)

in_features = X_train_scaled.shape[1]
net = tt.practical.MLPVanilla(
    in_features,
    [32, 16],
    out_features=1,
    batch_norm=True,
    dropout=0.2,
    activation=nn.SELU
)

model_deep = CoxPH(net, tt.optim.Adam(lr=0.002, weight_decay=1e-3))
callbacks = [tt.callbacks.EarlyStopping(patience=20)]

log = model_deep.fit(
    X_train_scaled, y_train_deep,
    batch_size=64,
    epochs=200,
    callbacks=callbacks,
    val_data=val_data,
    verbose=False
)

risk_train_deep = model_deep.predict(X_train_scaled).flatten()
risk_test_deep = model_deep.predict(X_test_scaled).flatten()

if risk_test_deep[e_test == 1].mean() < risk_test_deep[e_test == 0].mean():
    risk_train_deep = -risk_train_deep
    risk_test_deep = -risk_test_deep

deepsurv_train_c = concordance_index_censored(e_train.astype(bool), d_train, risk_train_deep)[0]
deepsurv_test_c = concordance_index_censored(e_test.astype(bool), d_test, risk_test_deep)[0]

# =====================================================================
# 6. Final Performance Benchmark Table (Pure Raw Decomposition)
# =====================================================================
summary_data = {
    "Model": ["Cox Proportional Hazards (CPH)", "Random Survival Forest (RSF)", "DeepSurv (Neural Network)"],
    "Model Family": ["Semiparametric / Linear", "Nonlinear Tree Ensemble", "Deep Neural Network"],
    "Train C-Index": [f"{cph_train_c:.4f}", f"{rsf_train_c:.4f}", f"{deepsurv_train_c:.4f}"],
    "Test C-Index": [f"{cph_test_c:.4f}", f"{rsf_test_c:.4f}", f"{deepsurv_test_c:.4f}"]
}

df_results_pure_raw = pd.DataFrame(summary_data)

print("\n" + "="*70)
print(f"📊 [Cirrhosis Exact Raw Decomposition] Benchmark (Features: {in_features})")
print("="*70)
print(df_results_pure_raw.to_string(index=False))
print("="*70)

# Test Models - German Credit Dataset

In [ ]:
# =====================================================================
# 0. Set random seed for reproducibility
# =====================================================================
SEED = 48764388
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =====================================================================
# 1. Fair Raw Decomposition (Extracting only 10 raw variables directly fed into formula calculations)
# =====================================================================
df_fair_german = df_german.copy()

# 1-1. Target Variables
df_fair_german["Duration (Months) (T)"] = df_fair_german["Duration_in_month"].astype("float32")
df_fair_german["Event (δ)"] = (df_fair_german["Credit_Risk"] == 2).astype("float32")

# 1-2. Categorical ordinal scores for paper formula mapping (Preserving ordinal scale conversion)
cs_map = {"A11": 1.0, "A12": 2.0, "A13": 3.0, "A14": 4.0}
ss_map = {"A61": 1.0, "A62": 2.0, "A63": 3.0, "A64": 4.0, "A65": 1.0}
es_map = {"A71": 0.0, "A72": 1.0, "A73": 2.0, "A74": 3.0, "A75": 4.0}
ps_map = {"A121": 4.0, "A122": 3.0, "A123": 2.0, "A124": 1.0}
gs_map = {"A101": 1.0, "A102": 2.0, "A103": 3.0}

df_fair_german["CS"] = df_fair_german["Status_Checking_Acc"].map(cs_map).fillna(1.0)
df_fair_german["SS"] = df_fair_german["Savings_Account"].map(ss_map).fillna(1.0)
df_fair_german["ES"] = df_fair_german["Present_Employment_Since"].map(es_map).fillna(0.0)
df_fair_german["PS"] = df_fair_german["Property"].map(ps_map).fillna(1.0)
df_fair_german["GS"] = df_fair_german["Other_Debtors"].map(gs_map).fillna(1.0)

# 1-3. List of 10 pure raw variables used in calculations (excluding unrelated variables)
raw_decomposed_cols = [
    # Numerical raw variables (5)
    "Credit_Amount",
    "Installment_Rate",
    "Age_in_years",
    "Present_Residence_Since",
    "Num_People_Liable",
    # Ordinal raw scores used in formulas (5)
    "CS",
    "SS",
    "ES",
    "PS",
    "GS"
]

target_cols = ["Duration (Months) (T)", "Event (δ)"]
all_cols = target_cols + raw_decomposed_cols

df_fair_german = df_fair_german[all_cols].dropna().reset_index(drop=True)

for col in raw_decomposed_cols:
    df_fair_german[col] = df_fair_german[col].astype("float32")

# =====================================================================
# 2. Stratified Data Splitting & Feature Scaling (Identical splitting)
# =====================================================================
feature_cols = raw_decomposed_cols
X = df_fair_german[feature_cols].values.astype("float32")
durations = df_fair_german["Duration (Months) (T)"].values.astype("float32")
events = df_fair_german["Event (δ)"].values.astype("float32")

# Train+Val (85%) vs Test (15%)
X_train_full, X_test, d_train_full, d_test, e_train_full, e_test = train_test_split(
    X, durations, events, test_size=0.15, random_state=SEED, stratify=events
)

# Train (70%) vs Val (15%)
X_train, X_val, d_train, d_val, e_train, e_val = train_test_split(
    X_train_full, d_train_full, e_train_full, test_size=0.17647, random_state=SEED, stratify=e_train_full
)

# Standardize features (fit based on training data)
scaler = StandardScaler()
X_train_full_scaled = scaler.fit_transform(X_train_full)
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# =====================================================================
# 3. [Model 1] Cox Proportional Hazards (CPH)
# =====================================================================
df_train_cph = pd.DataFrame(X_train_full_scaled, columns=feature_cols)
df_train_cph['t'] = d_train_full
df_train_cph['e'] = e_train_full

df_test_cph = pd.DataFrame(X_test_scaled, columns=feature_cols)
df_test_cph['t'] = d_test
df_test_cph['e'] = e_test

cph = CoxPHFitter(penalizer=0.05)
cph.fit(df_train_cph, duration_col='t', event_col='e')

cph_train_risk = cph.predict_log_partial_hazard(df_train_cph).values.flatten()
cph_test_risk = cph.predict_log_partial_hazard(df_test_cph).values.flatten()

cph_train_c = concordance_index_censored(e_train_full.astype(bool), d_train_full, cph_train_risk)[0]
cph_test_c = concordance_index_censored(e_test.astype(bool), d_test, cph_test_risk)[0]

# =====================================================================
# 4. [Model 2] Random Survival Forest (RSF)
# =====================================================================
y_train_struct = np.array(
    list(zip(e_train_full.astype(bool), d_train_full)),
    dtype=[('Status', '?'), ('Survival_in_days', '<f8')]
)
y_test_struct = np.array(
    list(zip(e_test.astype(bool), d_test)),
    dtype=[('Status', '?'), ('Survival_in_days', '<f8')]
)

rsf = RandomSurvivalForest(
    n_estimators=100,
    min_samples_split=10,
    min_samples_leaf=15,
    max_features="sqrt",
    n_jobs=-1,
    random_state=SEED
)
rsf.fit(X_train_full_scaled, y_train_struct)

rsf_train_c = rsf.score(X_train_full_scaled, y_train_struct)
rsf_test_c = rsf.score(X_test_scaled, y_test_struct)

# =====================================================================
# 5. [Model 3] DeepSurv (Deep Neural Network)
# =====================================================================
y_train_deep = (d_train, e_train)
y_val_deep = (d_val, e_val)
val_data = (X_val_scaled, y_val_deep)

in_features = X_train_scaled.shape[1]
net = tt.practical.MLPVanilla(
    in_features,
    [32, 16],
    out_features=1,
    batch_norm=True,
    dropout=0.2,
    activation=nn.SELU
)

model_deep = CoxPH(net, tt.optim.Adam(lr=0.002, weight_decay=1e-3))
callbacks = [tt.callbacks.EarlyStopping(patience=20)]

log = model_deep.fit(
    X_train_scaled, y_train_deep,
    batch_size=64,
    epochs=200,
    callbacks=callbacks,
    val_data=val_data,
    verbose=False
)

risk_train_deep = model_deep.predict(X_train_scaled).flatten()
risk_test_deep = model_deep.predict(X_test_scaled).flatten()

deepsurv_train_c = concordance_index_censored(e_train.astype(bool), d_train, risk_train_deep)[0]
deepsurv_test_c = concordance_index_censored(e_test.astype(bool), d_test, risk_test_deep)[0]

# =====================================================================
# 6. Final Performance Benchmark Table
# =====================================================================
summary_data = {
    "Model": ["Cox Proportional Hazards (CPH)", "Random Survival Forest (RSF)", "DeepSurv (Neural Network)"],
    "Model Family": ["Semiparametric / Linear", "Nonlinear Tree Ensemble", "Deep Neural Network"],
    "Train C-Index": [f"{cph_train_c:.4f}", f"{rsf_train_c:.4f}", f"{deepsurv_train_c:.4f}"],
    "Test C-Index": [f"{cph_test_c:.4f}", f"{rsf_test_c:.4f}", f"{deepsurv_test_c:.4f}"]
}

df_results_fair_german = pd.DataFrame(summary_data)

print("\n" + "="*70)
print(f"📊 [German Credit Exact Raw Decomposition] Benchmark (Features: {in_features})")
print("="*70)
print(df_results_fair_german.to_string(index=False))
print("="*70)

# LLM-as-a-Judge

In [ ]:
# !pip install google-genai pydantic

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sksurv.metrics import concordance_index_censored
import torchtuples as tt
from pycox.models import CoxPH

from google import genai
from google.genai import types
from pydantic import BaseModel, Field

# =====================================================================
# 0. Set random seed for reproducibility
# =====================================================================
SEED = 48764388
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =====================================================================
# 1. Feature Engineering and Multivariate Preprocessing
# =====================================================================
df_ch_wt = df_ch_dum.copy()

# 1-1. Generate nonlinear Stage-Weighted Risk feature
alpha = np.log(2)
stage_weight = np.exp(alpha * (df_ch_wt["Stage"] - 1.0))
biomarker_risk = (
    np.log(df_ch_wt["Cholesterol"] + 1.0)
    + np.log(df_ch_wt["Tryglicerides"] + 1.0)
    - np.log(df_ch_wt["Albumin"] + 1e-5)
)
df_ch_wt["Stage_Weighted_Risk"] = stage_weight * biomarker_risk

# 1-2. Define final 10-feature set for Cirrhosis
columns_to_keep = [
    "N_Days", "Status", "Ascites_Y", "Hepatomegaly_Y", "Spiders_Y",
    "Stage_Weighted_Risk", "Bilirubin", "Copper", "Prothrombin",
    "Alk_Phos", "SGOT", "Platelets"
]

df_ch_final = df_ch_wt[[c for c in columns_to_keep if c in df_ch_wt.columns]].copy()

# 1-3. Binarize Status (D: 1.0, C/CL: 0.0) and handle missing values
df_ch_final["Status"] = df_ch_final["Status"].map({"D": 1.0, "C": 0.0, "CL": 0.0})
df_ch_final = df_ch_final.dropna().reset_index(drop=True)
df_ch_final["Status"] = df_ch_final["Status"].astype("float32")
df_ch_final["N_Days"] = df_ch_final["N_Days"].astype("float32")

# =====================================================================
# 2. Stratified Data Splitting and Feature Scaling
# =====================================================================
feature_cols = [col for col in df_ch_final.columns if col not in ["N_Days", "Status"]]
X = df_ch_final[feature_cols].values.astype("float32")
durations = df_ch_final["N_Days"].values.astype("float32")
events = df_ch_final["Status"].values.astype("float32")

# Step 1: Split into Train+Val (85%) vs Test (15%)
X_train_full, X_test, d_train_full, d_test, e_train_full, e_test = train_test_split(
    X, durations, events, test_size=0.15, random_state=SEED, stratify=events
)

# Step 2: Split Train+Val into Train (70%) vs Val (15%) - For DeepSurv Early Stopping
X_train, X_val, d_train, d_val, e_train, e_val = train_test_split(
    X_train_full, d_train_full, e_train_full, test_size=0.17647, random_state=SEED, stratify=e_train_full
)

# Standardize features (Fit on Train only to prevent data leakage)
scaler = StandardScaler()
X_train_full_scaled = scaler.fit_transform(X_train_full)
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# =====================================================================
# 3. [Model 3] DeepSurv (Deep Neural Network) Training
# =====================================================================
y_train_deep = (d_train, e_train)
y_val_deep = (d_val, e_val)
val_data = (X_val_scaled, y_val_deep)

in_features = X_train_scaled.shape[1]
net = tt.practical.MLPVanilla(
    in_features, [32, 16], out_features=1,
    batch_norm=True, dropout=0.2, activation=nn.SELU
)

model_deep = CoxPH(net, tt.optim.Adam(lr=0.002, weight_decay=1e-3))
callbacks = [tt.callbacks.EarlyStopping(patience=20)]

log = model_deep.fit(
    X_train_scaled, y_train_deep,
    batch_size=64, epochs=100, callbacks=callbacks,
    val_data=val_data, verbose=False
)

# Predict risk scores and calibrate risk score direction
risk_train_deep = model_deep.predict(X_train_scaled).flatten()
risk_test_deep = model_deep.predict(X_test_scaled).flatten()

if risk_test_deep[e_test == 1].mean() < risk_test_deep[e_test == 0].mean():
    risk_train_deep = -risk_train_deep
    risk_test_deep = -risk_test_deep

deepsurv_test_c = concordance_index_censored(e_test.astype(bool), d_test, risk_test_deep)[0]
print(f"📊 DeepSurv Test C-Index: {deepsurv_test_c:.4f}")

# =====================================================================
# 4. LLM-as-a-Judge Integration (Gemini API)
# =====================================================================
# Set Gemini API Key (Enter your actual key or use Colab Secrets)
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

# Define Pydantic schema reflecting the 3 evaluation dimensions
class ComprehensiveEvaluationResult(BaseModel):
    clinical_interpretability_score: int = Field(description="Score 1-5: Does the risk score align logically with clinical features like Bilirubin and Stage-Weighted Risk?")
    risk_stratification_score: int = Field(description="Score 1-5: Is the risk tiering appropriate for prioritizing patient intervention?")
    actionable_safety_score: int = Field(description="Score 1-5: Is the output safe for clinical pre-screening without hallucination?")
    detailed_reasoning: str = Field(description="Comprehensive analysis of the model output.")
    is_safe_for_expert_review: bool = Field(description="Final verdict: Safe to pass to a human clinician? (True/False)")

def evaluate_deepsurv_output_with_gemini(patient_id: int, features_dict: dict, predicted_risk: float) -> dict:
    """
    Validates the DeepSurv model prediction for a specific patient using Gemini as an LLM judge.
    """
    system_instruction = """
    You are a strict and objective domain expert judge responsible for validating the outputs of a cirrhosis survival analysis model.
    Evaluate the model's risk prediction based on the provided patient clinical features across three dimensions:
    1) Clinical Interpretability, 2) Risk Stratification Logic, and 3) Actionable Safety.
    """

    user_content = f"""
    [Patient Case #{patient_id}]
    - Clinical Features: {json.dumps(features_dict)}
    - DeepSurv Predicted Risk Score: {predicted_risk:.4f}
    """

    try:
        # Request structured JSON response from Gemini
        response = client.models.generate_content(
            model='gemini-3.8-flash',
            contents=user_content,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                response_mime_type="application/json",
                response_schema=ComprehensiveEvaluationResult,
                temperature=0.0
            ),
        )
        return json.loads(response.text)
    except Exception as e:
        return {"error": str(e)}

# =====================================================================
# 5. Run Automated LLM Judge on Highest Risk Test Patient
# =====================================================================
print("\n--- Running LLM-as-a-Judge on Clean Cirrhosis High-Risk Prediction ---")

# Extract the patient with the highest predicted risk from the test set
highest_risk_idx = np.argmax(risk_test_deep)
sample_features = {col: float(X_test[highest_risk_idx][i]) for i, col in enumerate(feature_cols)}
sample_risk = float(risk_test_deep[highest_risk_idx])

# Execute Gemini Judge evaluation
evaluation_result = evaluate_deepsurv_output_with_gemini(
    patient_id=highest_risk_idx,
    features_dict=sample_features,
    predicted_risk=sample_risk
)

# Print formatted JSON result
print(json.dumps(evaluation_result, indent=4, ensure_ascii=False))